# Stage 5: CNN & ViT Baselines (B1, B2, B4)
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install kagglehub -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json, time
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T, torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, roc_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi']=120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{DEVICE} Dataset:{INPUT_DIR}')
CONFIG = {'bs':32,'epochs':30,'lr':1e-4,'wd':1e-4,'patience':8,'n_per_gen':200}

In [ ]:
# Cell 2: Data
meta_files = glob.glob(os.path.join(INPUT_DIR,'**','metadata.csv'), recursive=True)
all_dfs=[]
for mf in meta_files:
    df=pd.read_csv(mf); gd=os.path.dirname(mf)
    df['generator']=os.path.basename(gd)
    df['image_path']=df['image_path'].apply(lambda p:os.path.join(gd,p) if not os.path.isabs(p) else p)
    all_dfs.append(df)
if not all_dfs:
    exts={'.png','.jpg','.jpeg'}; recs=[]
    for root,_,files in os.walk(INPUT_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in exts:
                full=os.path.join(root,f); parts=os.path.relpath(root,INPUT_DIR).lower().split(os.sep)
                recs.append({'image_path':full,'target':0 if 'real' in parts else 1,'generator':parts[-1]})
    mdf=pd.DataFrame(recs)
else: mdf=pd.concat(all_dfs,ignore_index=True)
sampled=[gdf.sample(n=min(len(gdf),CONFIG['n_per_gen']),random_state=42) for _,gdf in mdf.groupby('generator')]
mdf=pd.concat(sampled,ignore_index=True)
mdf=mdf[mdf['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Images:{len(mdf)} Real:{(mdf["target"]==0).sum()} Fake:{(mdf["target"]==1).sum()}')
tr_v,te=train_test_split(mdf,test_size=0.2,stratify=mdf['target'],random_state=42)
tr,va=train_test_split(tr_v,test_size=0.15,stratify=tr_v['target'],random_state=42)
print(f'Train:{len(tr)} Val:{len(va)} Test:{len(te)}')

class RGBDs(Dataset):
    def __init__(s,df,tf=None): s.df=df.reset_index(drop=True);s.tf=tf
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; img=cv2.imread(r['image_path'],cv2.IMREAD_COLOR)
        if img is None: img=np.zeros((224,224,3),dtype=np.uint8)
        img=Image.fromarray(cv2.cvtColor(cv2.resize(img,(224,224)),cv2.COLOR_BGR2RGB))
        return s.tf(img) if s.tf else T.ToTensor()(img), int(r['target'])

class MagDs(Dataset):
    _LR,_LG,_LB=0.2989,0.5870,0.1140
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; img=cv2.imread(r['image_path'],cv2.IMREAD_COLOR)
        if img is None: return torch.zeros(3,224,224),int(r['target'])
        rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB).astype(np.float64)
        g=s._LR*rgb[:,:,0]+s._LG*rgb[:,:,1]+s._LB*rgb[:,:,2]
        g=cv2.resize(g,(224,224),interpolation=cv2.INTER_CUBIC)
        fft=np.fft.fftshift(np.fft.fft2(g)); mag=np.log1p(np.abs(fft))
        mag=(mag-mag.min())/(mag.max()-mag.min()+1e-8)
        return torch.FloatTensor(np.stack([mag]*3).astype(np.float32)),int(r['target'])

ttf=T.Compose([T.RandomHorizontalFlip(),T.RandomRotation(10),T.ColorJitter(0.1,0.1),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])
vtf=T.Compose([T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])

In [ ]:
# Cell 3: Trainer
def build_resnet():
    m=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc=nn.Sequential(nn.Dropout(0.3),nn.Linear(m.fc.in_features,1)); return m
def build_vit():
    m=models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    m.heads=nn.Sequential(nn.Dropout(0.3),nn.Linear(m.heads[0].in_features,1)); return m

def train_eval(model,trl,val,tel,name,cfg):
    model=model.to(DEVICE); crit=nn.BCEWithLogitsLoss()
    opt=optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['wd'])
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=cfg['epochs'])
    best,pat=0,0
    for ep in tqdm(range(cfg['epochs']),desc=name):
        model.train()
        for X,y in trl: X,y=X.to(DEVICE),y.float().to(DEVICE); opt.zero_grad(); crit(model(X).squeeze(-1),y).backward(); opt.step()
        sched.step()
        model.eval(); vp,vt=[],[]
        with torch.no_grad():
            for X,y in val: vp.append(torch.sigmoid(model(X.to(DEVICE)).squeeze(-1)).cpu().numpy()); vt.append(y.numpy())
        vp,vt=np.concatenate(vp),np.concatenate(vt); va=roc_auc_score(vt,vp) if len(np.unique(vt))>1 else 0
        if va>best: best=va;pat=0;torch.save(model.state_dict(),os.path.join(MODEL_DIR,f'{name}_best.pth'))
        else:
            pat+=1
            if pat>=cfg['patience']: print(f'Early stop {name} ep {ep+1}'); break
    model.load_state_dict(torch.load(os.path.join(MODEL_DIR,f'{name}_best.pth'),weights_only=True))
    model.eval(); tp,tt=[],[]
    with torch.no_grad():
        for X,y in tel: tp.append(torch.sigmoid(model(X.to(DEVICE)).squeeze(-1)).cpu().numpy()); tt.append(y.numpy())
    tp,tt=np.concatenate(tp),np.concatenate(tt)
    acc=accuracy_score(tt,(tp>0.5).astype(int)); auc=roc_auc_score(tt,tp)
    np2=sum(p.numel() for p in model.parameters())
    print(f'{name}: Acc={acc:.4f} AUC={auc:.4f} Params={np2:,}')
    print(classification_report(tt,(tp>0.5).astype(int),target_names=['Real','Fake']))
    return {'model':name,'test_accuracy':float(acc),'test_auc':float(auc),'n_parameters':np2,'preds':tp,'targets':tt}

In [ ]:
# Cell 4: Train
nw=2
rtrl=DataLoader(RGBDs(tr,ttf),batch_size=32,shuffle=True,num_workers=nw)
rval=DataLoader(RGBDs(va,vtf),batch_size=32,num_workers=nw)
rtel=DataLoader(RGBDs(te,vtf),batch_size=32,num_workers=nw)
mtrl=DataLoader(MagDs(tr),batch_size=32,shuffle=True,num_workers=nw)
mval=DataLoader(MagDs(va),batch_size=32,num_workers=nw)
mtel=DataLoader(MagDs(te),batch_size=32,num_workers=nw)

R={}
print('='*50+'\nB1: ResNet-50 RGB\n'+'='*50)
R['B1']=train_eval(build_resnet(),rtrl,rval,rtel,'ResNet50-RGB(B1)',CONFIG)
print('='*50+'\nB2: ResNet-50 Mag\n'+'='*50)
R['B2']=train_eval(build_resnet(),mtrl,mval,mtel,'ResNet50-Mag(B2)',CONFIG)
print('='*50+'\nB4: ViT-B/16\n'+'='*50)
R['B4']=train_eval(build_vit(),rtrl,rval,rtel,'ViT-B16-RGB(B4)',CONFIG)

In [ ]:
# Cell 5: Compare
for fn,k in [('results_kan.json','KAN'),('results_b3.json','B3')]:
    p=os.path.join(MODEL_DIR,fn)
    if os.path.exists(p):
        with open(p) as f: d=json.load(f)
        R[k]={'model':d['model'],'test_accuracy':d['test_accuracy'],'test_auc':d['test_auc'],'n_parameters':d['n_parameters']}
rows=[{'Model':r['model'],'AUC':r['test_auc'],'Accuracy':r['test_accuracy'],'Params':r['n_parameters']} for r in R.values()]
cdf=pd.DataFrame(rows).sort_values('AUC',ascending=False)
print('\nFULL COMPARISON:'); print(cdf.to_string(index=False))

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5)); fig.suptitle('Baseline Comparison',fontweight='bold')
cols=['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']
a1.barh(cdf['Model'],cdf['AUC'],color=cols[:len(cdf)]); a1.set_xlabel('AUC'); a1.set_xlim(0,1.05)
a2.scatter(cdf['Params'],cdf['AUC'],s=100,c=cols[:len(cdf)],zorder=5)
for _,r in cdf.iterrows(): a2.annotate(r['Model'][:15],(r['Params'],r['AUC']),fontsize=7,xytext=(5,5),textcoords='offset points')
a2.set_xlabel('Params');a2.set_ylabel('AUC');a2.set_xscale('log')
plt.tight_layout();plt.show()

fig,ax=plt.subplots(figsize=(8,6))
for k,c in [('B1','#3498db'),('B2','#e67e22'),('B4','#9b59b6')]:
    if k in R and 'preds' in R[k]:
        r=R[k]; fpr,tpr,_=roc_curve(r['targets'],r['preds'])
        ax.plot(fpr,tpr,label=f"{r['model']} ({r['test_auc']:.3f})",lw=2,color=c)
ax.plot([0,1],[0,1],'k--',alpha=0.3);ax.set_title('ROC Curves');ax.legend(loc='lower right')
plt.tight_layout();plt.show()

sv={k:{kk:v for kk,v in r.items() if kk not in ('preds','targets')} for k,r in R.items()}
with open(os.path.join(MODEL_DIR,'all_results.json'),'w') as f: json.dump(sv,f,indent=2)
cdf.to_csv(os.path.join(MODEL_DIR,'all_baselines_comparison.csv'),index=False)
print('All saved.')